#Our own custom data (non instrcution data) for domain specific finetuning
By Using PEFT LORA


In [5]:
!pip install -U peft bitsandbytes transformers accelerate trl datasets PyMuPDF

In [6]:
import fitz  # correct import (PyMuPDF)
import re
import torch
import gc
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType


In [7]:
# =========================
# 3. Clear GPU Memory
# =========================
gc.collect()
torch.cuda.empty_cache()

In [8]:
# =========================
# 4. Extract Text from PDF
# =========================
def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [9]:
# Provide correct path
pdf_path = "/content/Metformin.pdf"
pdf_texts = extract_text_from_pdf(pdf_path)

In [10]:
# =========================
# 5. Split into Paragraphs
# =========================
def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:
                paragraphs.append(clean)
    return paragraphs

paragraphs = split_paragraphs(pdf_texts)

In [30]:
paragraphs

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.',
 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA redu

In [11]:
# =========================
# 6. Create Dataset
# =========================
data = [{"text": p} for p in paragraphs]
dataset = Dataset.from_list(data)

print(dataset[0])  # check first row

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.'}


In [14]:
# =========================
# 7. Load Model & Tokenizer
# =========================
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

tokenizer = AutoTokenizer.from_pretrained(model_name)
# Fix pad token issue
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [15]:
# =========================
# 8. Tokenization
# =========================
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [17]:
# =========================
# 9. Load Model (8-bit FIXED)
# =========================

from transformers import BitsAndBytesConfig  # import quantization config

# create 8-bit config
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True  # enable 8-bit quantization
)

# load model with quantization config
model = AutoModelForCausalLM.from_pretrained(
    model_name,  # model name
    quantization_config=bnb_config,  # pass config instead of load_in_8bit
    device_map="auto"  # auto device placement
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [19]:
# =========================
# 10. Apply LoRA (PEFT)
# =========================
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(model, lora_config)

In [20]:
# =========================
# 11. Training Arguments
# =========================
training_args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [21]:
# =========================
# 12. Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)

In [22]:
# =========================
# 13. Train Model
# =========================
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=5, training_loss=9.481071472167969, metrics={'train_runtime': 11.5212, 'train_samples_per_second': 1.736, 'train_steps_per_second': 0.434, 'total_flos': 63629646888960.0, 'train_loss': 9.481071472167969, 'epoch': 5.0})

In [27]:
# =========================
# SAVE LoRA ADAPTER (VERY IMPORTANT)
# =========================

model.save_pretrained("./tinyllama-lora")  # save LoRA adapter properly
tokenizer.save_pretrained("./tinyllama-lora")  # save tokenizer

('./tinyllama-lora/tokenizer_config.json', './tinyllama-lora/tokenizer.json')

In [28]:
from peft import PeftModel

base_model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto"
)

# load adapter
trained_model = PeftModel.from_pretrained(
    base_model,
    "./tinyllama-lora"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [29]:
# =========================
# 15. Inference
# =========================
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe was more effective than Atorvastatin and placebo in reducing LDL-C. In addition, the data suggest that Ezetimibe is associated with less adverse events than Atorvastatin.
Pravastatinum is a combination of pravastatin (10 mg) with simvastatin (20 mg). Both statins have been used as first line therapy for cholesterol lowering in patients with hypercholest


# This is Explanation With Comments Each Lines For Undestandable

In [ ]:

# =========================
# 1. IMPORT REQUIRED LIBRARIES
# =========================

import fitz  # import PyMuPDF library (used to read and extract text from PDF files)
import re  # import regular expression module (used for splitting text into paragraphs)
import torch  # import PyTorch (used for model loading, training, and inference)
import gc  # import garbage collector (used to free unused memory)
from datasets import Dataset  # import Dataset class from HuggingFace to create dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer  # import transformer utilities
from peft import LoraConfig, get_peft_model, TaskType, PeftModel  # import LoRA (PEFT) related tools

# =========================
# 2. CLEAR GPU MEMORY
# =========================

gc.collect()  # manually trigger garbage collection to clear unused CPU memory
torch.cuda.empty_cache()  # clear unused GPU memory cache to prevent out-of-memory errors

# =========================
# 3. EXTRACT TEXT FROM PDF
# =========================

def extract_text_from_pdf(pdf_path):  # define function to extract text from a PDF file
    text_blocks = []  # create an empty list to store extracted text from each page
    with fitz.open(pdf_path) as doc:  # open the PDF file using PyMuPDF
        for page in doc:  # iterate over each page in the PDF
            text = page.get_text("text").strip()  # extract text from page and remove extra spaces
            if text:  # check if extracted text is not empty
                text_blocks.append(text)  # append cleaned text to list
    return text_blocks  # return list of page texts

pdf_path = "/content/Metformin.pdf"  # define path to your PDF file
pdf_texts = extract_text_from_pdf(pdf_path)  # call function to extract text from PDF

# =========================
# 4. SPLIT TEXT INTO PARAGRAPHS
# =========================

def split_paragraphs(pages):  # define function to split text into paragraphs
    paragraphs = []  # create empty list to store paragraphs
    for page_text in pages:  # loop through each page text
        chunks = re.split(r'\n\s*\n', page_text)  # split text based on double newlines
        for chunk in chunks:  # loop through each chunk (possible paragraph)
            clean = chunk.strip()  # remove extra spaces from chunk
            if len(clean) > 30:  # ignore very short text (less than 30 characters)
                paragraphs.append(clean)  # add valid paragraph to list
    return paragraphs  # return list of paragraphs

paragraphs = split_paragraphs(pdf_texts)  # convert extracted text into paragraphs

# =========================
# 5. CREATE DATASET
# =========================

data = [{"text": p} for p in paragraphs]  # convert each paragraph into dictionary format
dataset = Dataset.from_list(data)  # create HuggingFace dataset from list

print(dataset[0])  # print first data sample for verification

# =========================
# 6. LOAD MODEL & TOKENIZER
# =========================

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"  # define pretrained model name

tokenizer = AutoTokenizer.from_pretrained(model_name)  # load tokenizer for the model

if tokenizer.pad_token is None:  # check if pad token is missing
    tokenizer.pad_token = tokenizer.eos_token  # assign end-of-sequence token as pad token

# =========================
# 7. TOKENIZATION
# =========================

def tokenize_fn(examples):  # define function to tokenize dataset
    tokens = tokenizer(  # tokenize input text
        examples["text"],  # take text column
        truncation=True,  # truncate long sequences
        padding="max_length",  # pad sequences to max length
        max_length=512  # set maximum sequence length
    )
    tokens["labels"] = tokens["input_ids"].copy()  # set labels same as input_ids for training
    return tokens  # return tokenized data

tokenized = dataset.map(tokenize_fn, batched=True)  # apply tokenization to entire dataset

# =========================
# 8. LOAD MODEL (8-BIT MODE)
# =========================

from transformers import BitsAndBytesConfig  # import quantization configuration class

bnb_config = BitsAndBytesConfig(  # create configuration for 8-bit loading
    load_in_8bit=True  # enable 8-bit quantization to reduce memory usage
)

model = AutoModelForCausalLM.from_pretrained(  # load pretrained causal language model
    model_name,  # model name
    quantization_config=bnb_config,  # apply 8-bit quantization config
    device_map="auto"  # automatically assign model to GPU/CPU
)

# =========================
# 9. APPLY LoRA (PEFT)
# =========================

lora_config = LoraConfig(  # define LoRA configuration
    task_type=TaskType.CAUSAL_LM,  # specify task type
    r=8,  # rank of LoRA matrices (controls compression)
    lora_alpha=16,  # scaling factor for LoRA
    target_modules=["q_proj", "v_proj"],  # apply LoRA to attention layers
    lora_dropout=0.05,  # dropout to prevent overfitting
    bias="none"  # no bias training
)

model = get_peft_model(model, lora_config)  # attach LoRA adapters to model

# =========================
# 10. TRAINING ARGUMENTS
# =========================

training_args = TrainingArguments(  # define training configuration
    output_dir="./tinyllama-lora",  # directory to save model checkpoints
    num_train_epochs=5,  # number of training epochs
    per_device_train_batch_size=1,  # batch size per device
    gradient_accumulation_steps=8,  # accumulate gradients for larger effective batch
    learning_rate=2e-4,  # learning rate
    fp16=True,  # enable mixed precision training
    logging_steps=20,  # log every 20 steps
    save_total_limit=1,  # keep only latest checkpoint
    report_to="none"  # disable external logging
)

# =========================
# 11. TRAINER SETUP
# =========================

trainer = Trainer(  # initialize Trainer class
    model=model,  # model to train
    args=training_args,  # training configuration
    train_dataset=tokenized  # dataset for training
)

# =========================
# 12. TRAIN MODEL
# =========================

trainer.train()  # start training process

# =========================
# 13. SAVE LoRA ADAPTER
# =========================

model.save_pretrained("./tinyllama-lora")  # save LoRA adapter weights
tokenizer.save_pretrained("./tinyllama-lora")  # save tokenizer for reuse

# =========================
# 14. LOAD BASE MODEL
# =========================

base_model = AutoModelForCausalLM.from_pretrained(  # load original base model
    model_name,  # base model name
    device_map="auto"  # auto assign device
)

# =========================
# 15. LOAD LoRA ADAPTER
# =========================

trained_model = PeftModel.from_pretrained(  # load LoRA adapter
    base_model,  # base model
    "./tinyllama-lora"  # adapter path
)

# =========================
# 16. INFERENCE
# =========================

prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"  # define input prompt

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # tokenize input and move to GPU

outputs = trained_model.generate(  # generate text using trained model
    **inputs,  # pass tokenized input
    max_new_tokens=100,  # max tokens to generate
    temperature=0.8,  # randomness control
    top_p=0.9,  # nucleus sampling
    do_sample=True,  # enable sampling
    repetition_penalty=1.1  # reduce repetition
)

print("\nModel Output:\n")  # print heading
print(tokenizer.decode(outputs[0], skip_special_tokens=True))  # decode output tokens into text


#GitHub Push